In [26]:
import pandas as pd

url = "https://raw.githubusercontent.com/MainakRepositor/Datasets/master/AB_NYC_2019.csv"

df = pd.read_csv(url)

print(df.shape)

(48895, 16)


In [27]:
import pandas as pd

url = "https://raw.githubusercontent.com/MainakRepositor/Datasets/master/AB_NYC_2019.csv"

df = pd.read_csv(url)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Shape: (48895, 16)

Columns:
['id', 'name', 'host_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

Data types:
id                                  int64
name                               object
host_id                             int64
host_name                          object
neighbourhood_group                object
neighbourhood                      object
latitude                          float64
longitude                         float64
room_type                          object
price                               int64
minimum_nights                      int64
number_of_reviews                   int64
last_review                        object
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
dtype: object

Missing values:


In [28]:
print("===== RAW QUALITY REPORT =====")
print("\n1. Missing Values")
missing = df.isna().sum()
print(missing[missing > 0])
print("Total missing values:", missing.sum())
print("\n2. Duplicate Rows")
print("Duplicate rows:", df.duplicated().sum())
print("\n3. Impossible Values")

checks = {
    "Negative price": (df["price"] < 0).sum(),
    "Minimum nights <= 0": (df["minimum_nights"] <= 0).sum(),
    "Negative number of reviews": (df["number_of_reviews"] < 0).sum(),
    "Reviews per month < 0": (df["reviews_per_month"] < 0).sum(),
    "Availability below 0": (df["availability_365"] < 0).sum(),
    "Availability above 365": (df["availability_365"] > 365).sum(),
    "Invalid latitude": ((df["latitude"] < 40) | (df["latitude"] > 41)).sum(),
    "Invalid longitude": ((df["longitude"] < -75) | (df["longitude"] > -73)).sum()
}

for problem, count in checks.items():
    print(f"{problem}: {count}")

print("\n4. Inconsistent Units / Ranges")

print(
    "Price values expected in USD/night:",
    df["price"].min(), "to", df["price"].max()
)

print(
    "Minimum nights expected in nights:",
    df["minimum_nights"].min(), "to", df["minimum_nights"].max()
)

print(
    "Availability expected in days (0-365):",
    df["availability_365"].min(), "to", df["availability_365"].max()
)

print("\n5. Category Labels")

print("\nroom_type:")
print(df["room_type"].value_counts())

print("\nneighbourhood_group:")
print(df["neighbourhood_group"].value_counts())

valid_room_types = {
    "Entire home/apt",
    "Private room",
    "Shared room"
}

valid_boroughs = {
    "Bronx",
    "Brooklyn",
    "Manhattan",
    "Queens",
    "Staten Island"
}

print(
    "\nInvalid room_type labels:",
    (~df["room_type"].isin(valid_room_types)).sum()
)

print(
    "Invalid neighbourhood_group labels:",
    (~df["neighbourhood_group"].isin(valid_boroughs)).sum()
)

===== RAW QUALITY REPORT =====

1. Missing Values
name                    16
host_name               21
last_review          10052
reviews_per_month    10052
dtype: int64
Total missing values: 20141

2. Duplicate Rows
Duplicate rows: 0

3. Impossible Values
Negative price: 0
Minimum nights <= 0: 0
Negative number of reviews: 0
Reviews per month < 0: 0
Availability below 0: 0
Availability above 365: 0
Invalid latitude: 0
Invalid longitude: 0

4. Inconsistent Units / Ranges
Price values expected in USD/night: 0 to 10000
Minimum nights expected in nights: 1 to 1250
Availability expected in days (0-365): 0 to 365

5. Category Labels

room_type:
room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: int64

neighbourhood_group:
neighbourhood_group
Manhattan        21661
Brooklyn         20104
Queens            5666
Bronx             1091
Staten Island      373
Name: count, dtype: int64

Invalid room_type labels: 0
Invalid neighbourhood_group 

question 3


1. availability_rate = availability_365 / 365
Reason: Converts the number of available days into a yearly availability proportion.

2. minimum_stay_cost = price × minimum_nights
Reason: Estimates the minimum total cost=

3. reviews_per_year = reviews_per_month × 12
Reason: Converts monthly review activity into an annual measure.

4. host_listing_ratio = calculated_host_listings_count / (number_of_reviews + 1)
Reason: Relates the host's number of listings to the number

In [29]:
def load_data():
    url = "https://raw.githubusercontent.com/MainakRepositor/Datasets/master/AB_NYC_2019.csv"
    return pd.read_csv(url)

In [30]:
def clean(df):
    df = df.copy()

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Remove impossible values
    df = df[df["price"] > 0]
    df = df[df["minimum_nights"] > 0]
    df = df[df["availability_365"].between(0, 365)]

    # Fill missing text values
    df["name"] = df["name"].fillna("Unknown")
    df["host_name"] = df["host_name"].fillna("Unknown")

    # Convert date column
    df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")

    return df


In [31]:
def add_features(df):
    df = df.copy()

    df["availability_rate"] = df["availability_365"] / 365

    df["minimum_stay_cost"] = df["price"] * df["minimum_nights"]

    df["reviews_per_year"] = df["reviews_per_month"] * 12

    df["host_listing_ratio"] = (
        df["calculated_host_listings_count"] /
        (df["number_of_reviews"] + 1)
    )

    return df

In [32]:
def build_pipeline():
    numeric_features = [
        "minimum_nights",
        "number_of_reviews",
        "reviews_per_month",
        "calculated_host_listings_count",
        "availability_365",
        "availability_rate",
        "minimum_stay_cost",
        "reviews_per_year",
        "host_listing_ratio"
    ]

    categorical_features = [
        "neighbourhood_group",
        "neighbourhood",
        "room_type"
    ]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    pipeline = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return pipeline

In [33]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

df = load_data()

df = clean(df)

df = add_features(df)

pipeline = build_pipeline()

X = df.drop(columns=["price", "id", "name", "host_id", "host_name", "last_review"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_train = pipeline.fit_transform(X_train)

X_test = pipeline.transform(X_test)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (39107, 235)
X_test shape: (9777, 235)
y_train shape: (39107,)
y_test shape: (9777,)


In [34]:
assert not np.isnan(X_train.toarray() if hasattr(X_train, "toarray") else X_train).any()
assert not np.isnan(X_test.toarray() if hasattr(X_test, "toarray") else X_test).any()

assert X_train.shape[1] == X_test.shape[1]

X_train_dense = X_train.toarray() if hasattr(X_train, "toarray") else X_train
assert np.all(X_train_dense.std(axis=0) != 0)

print("All assertions passed.")
print("No NaN values remain.")
print("Train and test have the same number of columns.")
print("No column is constant.")

All assertions passed.
No NaN values remain.
Train and test have the same number of columns.
No column is constant.


In [35]:


raw_df = load_data()
clean_df = add_features(clean(raw_df.copy()))

before = {
    "Stage": "Raw",
    "Row Count": len(raw_df),
    "Column Count": raw_df.shape[1],
    "Missing Values": raw_df.isna().sum().sum(),
    "Duplicate Count": raw_df.duplicated().sum()
}

after = {
    "Stage": "Final",
    "Row Count": len(clean_df),
    "Column Count": clean_df.shape[1],
    "Missing Values": clean_df.isna().sum().sum(),
    "Duplicate Count": clean_df.duplicated().sum()
}

comparison = pd.DataFrame([before, after])

comparison

,Stage,Row Count,Column Count,Missing Values,Duplicate Count
0,Raw,48895,16,20141,0
1,Final,48884,20,30153,0


In [36]:
import joblib

clean_df.to_csv("cleaned_data.csv", index=False)
joblib.dump(pipeline, "pipeline.joblib")

print("Files saved successfully:")
print("cleaned_data.csv")
print("pipeline.joblib")

from google.colab import files

files.download("cleaned_data.csv")
files.download("pipeline.joblib")

Files saved successfully:
cleaned_data.csv
pipeline.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DECISION LOG

1. Removed duplicate rows — 0 duplicates were found, so no rows were removed for duplication.

2. Removed listings with price <= 0 — 0 invalid price records were found.

3. Removed listings with minimum_nights <= 0 — 0 invalid records were found.

4. Removed listings with availability_365 outside 0–365 — 0 invalid records were found.

5. Filled missing name values with "Unknown" — 16 missing values were found.

6. Filled missing host_name values with "Unknown" — 21 missing values were found.

7. Converted last_review to datetime — 10,052 missing values were found and retained as missing dates.

8. Imputed reviews_per_month using the median — 10,052 missing values were found.

9. Used median imputation for numerical features — chosen because numerical columns contain potential outliers.

10. Used most-frequent imputation for categorical features — chosen to preserve the most common category.

11. Applied one-hot encoding to categorical features — room_type has 3 categories and neighbourhood_group has 5 categories.

12. Applied standardization to numerical features — used to place numerical features on a comparable scale.

13. Used price as the target variable — price is a numerical variable suitable for a future prediction task.

14. Used an 80/20 train-test split — 80% training data and 20% testing data, with random_state=42.

15. No category labels were removed — 0 invalid room_type labels and 0 invalid neighbourhood_group labels were found.

question-10- Some issues still remain, such as extreme prices and very high minimum-stay values. Missing review dates also cannot be recovered accurately. These limitations could not be fixed without making assumptions about the original data.
